# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiment: **E010-anchors12** — one lever on top of E009 (0.803 holdout /
0.789 LB): slice coverage. At 8 anchors a ~30-slice series is touched at ~24
positions with gaps between anchors; findings living in skipped slices are
invisible regardless of the attention head. 12 anchors tile ~36 triplet positions
— effectively full coverage of a typical stack. This is our own strongest
unplayed evidence (E005: full-slice frozen attention beat sparse triplets +0.014)
and the corrected E009 audit's ~2.9:1 model-error:label-error ratio says model
levers still pay. Everything else frozen for a clean pair against 0.803.

GPU cost (single arm): decode 4 slots ~70 min (unchanged) + unified fine-tune at
~48 bag items/study (~+50% vs E009's 32) ~3-3.5h → **total ~4-4.5 T4 hours**.
Requires `WANDB_API_KEY`, `knee-labels`, internet.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
COMMIT = "f32d238"  # main: E009 code + report laterality-frame fix (no train-path changes since 5290655)
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_blended import BLENDED_LABEL_SOURCE

In [ ]:
# Competition data (DICOMs + series metadata) is pre-mounted; blended soft labels
# (with the per-cell __weight companions for E006a) via the knee-labels dataset;
# the E005 winner's feature bank + checkpoints via knee-e005-artifacts.
from pathlib import Path

from knee.data import load_blended_labels, weight_matrix

SLUG = "rsna-knee-abnormality-detection"
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)


def find_input(name: str, filename: str) -> Path:
    bases = [Path("/kaggle/input") / name, Path("/kaggle/input/datasets/josiemachalek") / name]
    for base in bases:
        if (base / filename).exists():
            return base / filename
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{name}/{filename} not found; mounts: {listing}")


labels = load_blended_labels(find_input("knee-labels", "blended_labels_v1.csv"), include_weights=True)
weights = weight_matrix(labels)
print(f"blended labels: {len(labels)} studies; weights {weights.shape}")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(
        project="rsna-knee",
        config={"commit": COMMIT, "label_source": BLENDED_LABEL_SOURCE, "n_label_studies": len(labels)},
    )
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E010 config = E009 with n_anchors 8 -> 12, nothing else moved. The 4th slot
# stays APPENDED — plane-embedding indices are positional, so order changes would
# silently remap planes on any warm start.
from knee.model import DEFAULT_BACKBONE

SERIES_TYPES = [
    SeriesType.SAGITTAL_FLUID,
    SeriesType.CORONAL_FLUID,
    SeriesType.AXIAL_FLUID,
    SeriesType.SAGITTAL_NONFLUID,  # E009: the T1 slot
]
BACKBONE = DEFAULT_BACKBONE
INPUT_SIZE = 224
CROP_MM = 140.0
USE_WEIGHTS = True
CANONICALIZE_LATERALITY = True  # E009: canonical right-knee frame, baked into the cache
E010_CONFIG = {"n_anchors": 12, "anchor_window": (0.1, 0.9), "frozen_epochs": 4, "epochs": 18}

CHECKPOINT_DIR = Path("/kaggle/working")
# Cache is geometry-keyed by dir name; E010 shares E009's exact decode geometry
# (224px, crop140, laterality on) but /tmp never survives between runs anyway.
CACHE_DIR = Path("/tmp/pixel_cache_e009")

In [ ]:
# Fixed 90/10 split — the fine-tune-era regime marker. Same seed always, so every
# fine-tune era experiment shares the split and stays comparable.
import numpy as np

from knee.cv import stratified_holdout
from knee.labels import LABEL_COLUMNS

label_matrix = labels[list(LABEL_COLUMNS)].to_numpy(dtype=np.float32)
val_mask = stratified_holdout(label_matrix, val_fraction=0.1, seed=0)
print(f"split: {int((~val_mask).sum())} train / {int(val_mask.sum())} val")

In [ ]:
# E010: pixel cache (~70 min for 4 slots), then ONE unified fine-tune (~3-3.5h at
# 12 anchors x up to 4 planes = up to 48 images per study). Sanity-read the
# laterality tally before trusting the run: expect roughly half left_mirrored,
# half right, ~97% resolved overall, and only a handful ambiguous (the bilateral
# series) or no_geometry — a skewed tally means the geometry read is broken.
from knee.finetune import FinetuneConfig, build_pixel_cache, finetune_unified
from knee.model import MultiPlaneModel

cache = build_pixel_cache(
    COMP_ROOT, labels, CACHE_DIR, series_types=SERIES_TYPES,
    input_size=INPUT_SIZE, crop_mm=CROP_MM,
    canonicalize_laterality=CANONICALIZE_LATERALITY,
)
print("cache coverage:", {t.value: n for t, n in cache.coverage.items()})
print("laterality:", {outcome.value: n for outcome, n in cache.laterality.items()})

model = MultiPlaneModel(BACKBONE, SERIES_TYPES)
result = finetune_unified(
    CACHE_DIR, label_matrix, val_mask,
    model=model,
    out_path=CHECKPOINT_DIR / "e010_unified.pt",
    config=FinetuneConfig(**E010_CONFIG),
    input_size=INPUT_SIZE, crop_mm=CROP_MM,
    cell_weights=weights if USE_WEIGHTS else None,
    label_source=BLENDED_LABEL_SOURCE,
    laterality_normalized=CANONICALIZE_LATERALITY,
)
print(f"E010 unified: best val macro {result.best_val_macro_auc:.3f} (epoch {result.best_epoch + 1})")
print({label: round(auc, 3) for label, auc in result.val_auc_per_label.items()})

In [ ]:
# e010_unified.pt persists as notebook output. Submission bar (team policy
# 2026-09-03): >=0.80 holdout macro AND beating E009's 0.803 justifies the
# publish->infer->submit path; below that, record and bank the learning. The
# checkpoint stamps laterality_normalized, so inference mirrors automatically.
import math

if run is not None:
    run.config.update({
        "backbone": BACKBONE, "input_size": INPUT_SIZE, "crop_mm": CROP_MM,
        "tier_weighted": USE_WEIGHTS, "model_kind": "multiplane",
        "n_planes": len(SERIES_TYPES), "laterality_normalized": CANONICALIZE_LATERALITY,
        **E010_CONFIG,
    })
    wandb.log({
        "holdout/unified_macro": result.best_val_macro_auc,
        **{f"holdout/unified/{label}": auc for label, auc in result.val_auc_per_label.items() if not math.isnan(auc)},
    })
    run.finish()